# 06 · Robustness controls and export

Controls for the two main results, each answering one alternative explanation:

1. **Flagged samples:** H1 without the 4 awakenings whose z-scale is set by an artifact elsewhere (NB1 section 8).
2. **Site:** H1 within the `subject` site alone (site is fully confounded with E1 availability and hardware).
3. **Artifact timing:** is E4 more artifact-laden than E2? (pre-waking movement)
4. **Artifact-only decoding:** can the per-epoch rejection rate alone decode report content? Should be at chance.

Then everything in `results/` on Drive is zipped for the GitHub repo.

In [1]:
# Bootstrap: same cell in every notebook. On Colab: mount Drive, get the repo, install extras.
# The repo is private: add a read-only GitHub token to Colab Secrets as GH_TOKEN.
import os, sys, subprocess, base64
REPO_URL = "https://github.com/sanjanasinghx/dream2image-es606.git"
if "google.colab" in sys.modules and not os.environ.get("D2I_HEADLESS"):  # headless = run by d2i.runner
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO = "/content/dream2image-es606"
    try:
        auth = base64.b64encode(f"x-access-token:{userdata.get('GH_TOKEN')}".encode()).decode()
        git = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]  # token never written to disk
    except Exception:
        git = ["git"]  # no secret: works once the repo is public
    if not os.path.exists(REPO):
        subprocess.run([*git, "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run([*git, "-C", REPO, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"], check=True)
    freeze = lambda: subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    before = freeze()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
    if freeze() != before:  # e.g. numpy upgraded under an already-loaded copy -> broken imports
        print("Packages changed: restarting the runtime. When it reconnects, Run all again.")
        os.kill(os.getpid(), 9)
else:
    REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from d2i import env
cfg = env.load_config()
rng = env.seed_everything(cfg["seed"])
print("data root:", env.data_root())

data root: <data root>


In [2]:
import json, shutil
import numpy as np, pandas as pd
from d2i import data, decode, stats
hc, dc = cfg["h1"], cfg["h2"]
der, res_dir = env.path(cfg, "derived"), env.path(cfg, "results")
tables = res_dir / "tables"; tables.mkdir(exist_ok=True)
step0 = json.loads((der / "step0.json").read_text())
roi = pd.read_csv(der / "features_roi.csv")
roi = roi[roi.epoch.isin(hc["epochs"])].assign(time30=lambda d: d.center_s / 30)
rows = []

## 1-2. H1 without flagged samples, and within one site

In [3]:
for label, d, cov in [
        ("all (primary)", roi, hc["covariates"]),
        ("without scaling-flagged", roi[~roi.sample_id.isin(step0["scaling_flagged"])], hc["covariates"]),
        ("subject site only", roi[roi.site == "subject"], ["reject_rate"])]:
    for y in hc["primary"]:
        rows.append({"control": label, **stats.fit_trend(d, y, cov)})
ctrl = pd.DataFrame(rows)
print(ctrl[["control", "y", "estimate", "ci_low", "ci_high", "p", "n_awakenings"]].round(4).to_string(index=False))

                control            y  estimate  ci_low  ci_high      p  n_awakenings
          all (primary) logpow_delta    0.0285 -0.0018   0.0588 0.0651           118
          all (primary)  ap_exponent    0.0430  0.0110   0.0751 0.0085           118
without scaling-flagged logpow_delta    0.0308  0.0012   0.0603 0.0415           114
without scaling-flagged  ap_exponent    0.0440  0.0125   0.0754 0.0062           114
      subject site only logpow_delta    0.0500  0.0166   0.0835 0.0034            96
      subject site only  ap_exponent    0.0704  0.0346   0.1062 0.0001            96


## 3. Does artifact load rise toward awakening?

In [4]:
rej = pd.read_csv(tables / "reject_rates.csv")
rej = rej[rej.epoch.isin(hc["epochs"])].assign(time30=lambda d: d.center_s / 30)
art = stats.fit_trend(rej, "reject_rate", ["site"])
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in art.items()})
ctrl = pd.concat([ctrl, pd.DataFrame([{"control": "artifact rate trend", **art}])])

{'y': 'reject_rate', 'term': 'time30', 'estimate': np.float64(0.0044), 'ci_low': -0.0323, 'ci_high': 0.0411, 'p': np.float64(0.8153), 'std_estimate': np.float64(0.0268), 'n_obs': 366, 'n_awakenings': 122, 'n_participants': 37, 'covariates': 'site', 'converged': True}


## 4. Artifact-only decoding control

In [5]:
long = pd.read_csv(der / "features_long.csv")
X, meta = decode.feature_matrices(long, dc["epochs"], dc["features"])
z = np.load(der / "report_embeddings.npz", allow_pickle=True)
E = dict(zip(z["sample_id"], z["emb"]))
Y = np.array([E[s] for s in meta.sample_id])
rr = rej.pivot_table(index="sample_id", columns="epoch", values="reject_rate").loc[meta.sample_id]
rng = np.random.default_rng(cfg["seed"])
perms = [rng.permutation(len(Y)) for _ in range(dc["n_perm"])]
for e in dc["epochs"]:
    folds = decode.make_folds(rr[[e]].to_numpy(), meta.participant.values)
    obs = decode.loso_rank(folds, Y, dc["alphas"])
    null = np.array([decode.loso_rank(folds, Y[p], dc["alphas"]) for p in perms])
    r = decode.vs_null(obs, null, {"control": "artifact-only decoding", "epoch": e})
    print(r)
    ctrl = pd.concat([ctrl, pd.DataFrame([r])])
ctrl.to_csv(tables / "controls.csv", index=False)

{'control': 'artifact-only decoding', 'epoch': 'E2', 'rank_acc': np.float64(0.529108910891089), 'null_mean': np.float64(0.49836257425742575), 'above_chance': np.float64(0.030746336633663274), 'null_p95': np.float64(0.5482326732673267), 'p_perm': np.float64(0.18581418581418582), 'n': 101}


{'control': 'artifact-only decoding', 'epoch': 'E3', 'rank_acc': np.float64(0.4504950495049505), 'null_mean': np.float64(0.49596297029702974), 'above_chance': np.float64(-0.04546792079207923), 'null_p95': np.float64(0.5461534653465346), 'p_perm': np.float64(0.906093906093906), 'n': 101}


{'control': 'artifact-only decoding', 'epoch': 'E4', 'rank_acc': np.float64(0.5285148514851484), 'null_mean': np.float64(0.5004092079207921), 'above_chance': np.float64(0.028105643564356297), 'null_p95': np.float64(0.549509900990099), 'p_perm': np.float64(0.1888111888111888), 'n': 101}


## 5. Export: zip `results/` for the GitHub repo

In [6]:
zip_path = shutil.make_archive(str(env.data_root() / "results_export"), "zip", res_dir)
env.write_manifest(cfg, "06_controls", [tables / "controls.csv"])
print("wrote", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
print("Download it from Drive (ES606_Dream2Image/results_export.zip) and unzip into the repo's results/ folder.")

wrote <data root>/results_export.zip (0.9 MB)
Download it from Drive (ES606_Dream2Image/results_export.zip) and unzip into the repo's results/ folder.
